# ACS ECG Detector — обучение на Google Colab (GPU T4)

Выполняет Этапы 4-5 (CNN + Multimodal) на бесплатном GPU T4.

## Перед запуском
1. Убедиться, что `processed.zip` загружен на Yandex.Disk (ссылка уже в коде)
2. Runtime -> Change runtime type -> T4 GPU

## Если сессия прервалась
Runtime -> Сменить среду выполнения -> T4 GPU -> **Run all**
Ячейка 1 проверит чекпоинты и продолжит с последней сохранённой эпохи.

In [ ]:
# Ячейка 1: Установка зависимостей
import sys, subprocess, os, glob

REPO_URL = "https://github.com/mlmamalyga83/acs-risk-ai.git"

# Клонировать репозиторий
if not os.path.exists('acs-risk-ai'):
    !git clone {REPO_URL}

%cd acs-risk-ai

# Обновить код (если уже склонирован — git pull)
!git fetch origin && git reset --hard origin/master 2>/dev/null || echo "first clone"

!pip install -r requirements.txt -q

# Проверить чекпоинты (для восстановления после сбоя)
ckpts = glob.glob('models/checkpoint_*.pt')
if ckpts:
    for c in ckpts:
        sz = os.path.getsize(c) / 1e6
        print(f'  Checkpoint found: {c} ({sz:.1f} MB)')
    print('Will resume from checkpoint if stages interrupted')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Ячейка 2: Скачать processed.zip с Yandex.Disk
import urllib.request, urllib.parse, json, os

# Пропустить, если данные уже есть
if os.path.exists('data/processed/X_val.npy'):
    print('Data already exists, skipping download')
else:
    PUBLIC_KEY = "https://disk.yandex.ru/d/03GsU4NnX2RpEQ"
    api_url = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={urllib.parse.quote(PUBLIC_KEY)}"
    print("Getting download link from Yandex.Disk...")
    with urllib.request.urlopen(api_url) as resp:
        download_url = json.loads(resp.read())["href"]
    !wget -O /content/processed.zip "{download_url}" --show-progress
    !unzip -o /content/processed.zip -d /content/
    print('OK processed.zip extracted to /content/')

# Скопировать данные в репозиторий
!cp -r /content/data/ ./ 2>/dev/null || echo "data ok"
!cp -r /content/config/ ./ 2>/dev/null || echo "config ok"
print('OK data ready')

In [ ]:
# Ячейка 3: Проверить GPU и данные
import torch
assert torch.cuda.is_available(), "GPU не обнаружен! Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

import numpy as np
proc = 'data/processed'
print(f"Train files: {open(f'{proc}/X_train_manifest.txt').read().count(chr(10))} batches")
print(f"Val:   {np.load(f'{proc}/X_val.npy', mmap_mode='r').shape}")
print(f"Test:  {np.load(f'{proc}/X_test.npy', mmap_mode='r').shape}")
print(f"Train: {np.load(f'{proc}/y_train.npy').shape}, ACS={np.load(f'{proc}/y_train.npy').mean()*100:.1f}%")
print('OK all checks passed')

In [ ]:
# Ячейка 5: CNN обучение (~1 мин на T4)
# Если был чекпоинт — продолжит с него (--resume)
!python run.py --stage cnn --resume

In [ ]:
# Ячейка 6: Multimodal обучение (~15 мин на T4)
# Если был чекпоинт — продолжит с него (--resume)
!python run.py --stage multimodal --resume

In [ ]:
# Ячейка 6: Сохранить результаты — упаковать и скачать
import os, tarfile

models = [f for f in os.listdir('models') if f.endswith('.pt')]
print(f'Models trained: {len(models)}')
for m in models:
    sz = os.path.getsize(f'models/{m}') / 1e6
    print(f'  {m}: {sz:.1f} MB')

with tarfile.open('/content/models.tar.gz', 'w:gz') as tar:
    for m in models:
        tar.add(f'models/{m}')
print('OK /content/models.tar.gz created')
print('Нажмите на папку /content/ (слева, иконка Files) -> models.tar.gz -> ПКМ -> Download')